# FASTopic avec UMAP

## Hypothèse
On applique UMAP avant l'entraînement de FASTopic afin de compresser les embeddings de documents, réduire le coût de calcul et tester si un espace plus compact aide aussi la séparation des topics.

## Choix expérimental
UMAP est appliqué **avant** `fit_transform`. Après entraînement, il ne servirait qu'à visualiser les documents et n'aurait aucun effet sur le temps de calcul ni sur la géométrie apprise par le modèle.


## 1) Install dependencies (Kaggle-safe)


In [1]:
import sys
import subprocess
import importlib.util
import pandas as pd


def ensure_package(import_name: str, pip_spec: str, no_deps: bool = False):
    if importlib.util.find_spec(import_name) is None:
        cmd = [sys.executable, '-m', 'pip', 'install', '--upgrade-strategy', 'only-if-needed', '--no-cache-dir']
        if no_deps:
            cmd.append('--no-deps')
        cmd.append(pip_spec)
        print('Installing', pip_spec)
        subprocess.check_call(cmd)


ensure_package('scipy', 'scipy')
ensure_package('yaml', 'pyyaml')
ensure_package('sentence_transformers', 'sentence-transformers')
ensure_package('fastopic', 'fastopic==1.0.1', no_deps=True)
ensure_package('gensim', 'gensim')
ensure_package('umap', 'umap-learn')

print('Dependency check OK (torch/cuda untouched)')


Installing fastopic==1.0.1
Dependency check OK (torch/cuda untouched)


In [2]:
print("numpy handled by dependency cell")


numpy handled by dependency cell


## 2) Imports and env


In [3]:
import os
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import scipy.io
import scipy.sparse as sp
import torch
import umap
import yaml
from sklearn import metrics
from fastopic import FASTopic
from sentence_transformers import SentenceTransformer


2026-03-11 11:13:37.944024: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773227618.177687      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773227618.240885      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773227618.745268      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773227618.745328      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773227618.745331      55 computation_placer.cc:177] computation placer alr

## 3) Dataset discovery

La cellule suivante contient la configuration et la detection automatique des datasets (local/Kaggle).


In [4]:
TARGET_DATASETS = ['20NG', 'AGNews', 'IMDB', 'YahooAnswer']
K_LIST = [20, 50, 100]
RANDOM_SEED = 42

REQUIRED_DATA_FILES = [
    'train_bow.npz', 'test_bow.npz', 'train_labels.txt', 'test_labels.txt', 'vocab.txt', 'word_embeddings.npz'
]

IS_KAGGLE = Path('/kaggle').exists()


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'ECTRM_source' / 'ECRTM' / 'data').exists():
            return candidate
    return start


if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working')
    INPUT_ROOT = Path('/kaggle/input')
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    INPUT_ROOT = PROJECT_ROOT


def has_required_files(path: Path) -> bool:
    return path.exists() and path.is_dir() and all((path / f).exists() for f in REQUIRED_DATA_FILES)


def guess_dataset_name(path: Path):
    s = str(path).lower()
    if '20ng' in s or '20news' in s:
        return '20NG'
    if 'agnews' in s or 'ag_news' in s:
        return 'AGNews'
    if 'yahoo' in s:
        return 'YahooAnswer'
    if 'imdb' in s:
        return 'IMDB'
    return None


def iter_candidate_dirs(base: Path, max_depth: int = 5):
    if not base.exists():
        return
    base_depth = len(base.parts)
    for root, dirs, _ in os.walk(base):
        root_path = Path(root)
        depth = len(root_path.parts) - base_depth
        if depth > max_depth:
            dirs[:] = []
            continue
        yield root_path


def discover_dataset_dirs():
    found = {}

    local_data_root = PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'data'
    for ds in TARGET_DATASETS:
        p = local_data_root / ds
        if has_required_files(p):
            found[ds] = p

    scan_roots = [Path('/kaggle/input'), PROJECT_ROOT, INPUT_ROOT] if IS_KAGGLE else [PROJECT_ROOT, INPUT_ROOT]
    seen = set()
    for scan_root in scan_roots:
        for cand in iter_candidate_dirs(scan_root):
            cs = str(cand)
            if cs in seen:
                continue
            seen.add(cs)
            if not has_required_files(cand):
                continue
            ds = guess_dataset_name(cand)
            if ds is not None and ds not in found:
                found[ds] = cand
    return found


DATASET_DIRS = discover_dataset_dirs()
print('IS_KAGGLE:', IS_KAGGLE)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('INPUT_ROOT:', INPUT_ROOT)
print('\nResolved datasets:')
for ds in TARGET_DATASETS:
    print('-', ds, DATASET_DIRS.get(ds))


IS_KAGGLE: True
PROJECT_ROOT: /kaggle/working
INPUT_ROOT: /kaggle/input

Resolved datasets:
- 20NG /kaggle/input/datasets/snlosnl/kaggleinputectrm-source-data-20ng
- AGNews /kaggle/input/datasets/snlosnl/ectrm-sourcedataagnews
- IMDB /kaggle/input/datasets/snlosnl/ectrm-sourcedataimdb
- YahooAnswer /kaggle/input/datasets/snlosnl/ectrm-yahooanswer


## 4) Config and helpers


In [5]:
MODEL_TAG = 'Fast_topic_avec_UMAP'
OUTPUT_DIR = (Path('/kaggle/working') / 'Sortie_mat' / 'output' / 'kaggle' / MODEL_TAG) if IS_KAGGLE else (PROJECT_ROOT / 'Sortie_mat' / MODEL_TAG)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FASTOPIC_DEFAULTS = {
    'epochs': 200,
    'learning_rate': 0.002,
    'low_memory': True,
    'normalize_embeddings': True,
    'doc_embed_model': 'all-MiniLM-L6-v2',
    'text_batch_size': 128,
    'min_low_memory_batch_size': 2000,
    'max_low_memory_batch_size': 4000,
    'dt_alpha_default': 10.0,
    'tw_alpha': 2.0,
    'theta_temp': 1.0,
}
DOC_EMBEDDING_MODE = 'auto'
DOC_MODE_BY_DATASET = {'20NG': 'text', 'IMDB': 'text', 'AGNews': 'bow_avg', 'YahooAnswer': 'bow_avg'}
DT_ALPHA_BY_DATASET = {'20NG': 8.0, 'IMDB': 8.0, 'AGNews': 12.0, 'YahooAnswer': 12.0}
UMAP_CONFIG = {'enabled': True, 'n_components': 64, 'n_neighbors': 15, 'min_dist': 0.0, 'metric': 'cosine', 'random_state': RANDOM_SEED, 'normalize_after': True}
FASTOPIC_DATA_CACHE = {}


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_cfg(path):
    if path is None or (not Path(path).exists()):
        return {}
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f) or {}


def discover_cfg_paths():
    cfg = {}
    for root in [PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'ECRTM' / 'configs' / 'model', PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'configs' / 'model']:
        for ds in TARGET_DATASETS:
            p = root / f'ECRTM_{ds}.yaml'
            if p.exists():
                cfg[ds] = p
    for ds in TARGET_DATASETS:
        if ds in cfg:
            continue
        name = f'ECRTM_{ds}.yaml'
        for base in [Path('/kaggle/input'), Path('/kaggle/working'), PROJECT_ROOT]:
            if not base.exists():
                continue
            for r, _, files in os.walk(base):
                if name in files:
                    cfg[ds] = Path(r) / name
                    break
            if ds in cfg:
                break
    return cfg


CONFIGS = discover_cfg_paths()


def load_vocab(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]


def load_bow_csr(path: Path):
    return sp.load_npz(path).astype(np.float32).tocsr()


def load_word_embeddings(path: Path, vocab_size=None):
    try:
        arr = sp.load_npz(path).astype(np.float32).toarray()
        if vocab_size is not None and arr.shape[0] != vocab_size and arr.shape[1] == vocab_size:
            arr = arr.T
        return arr
    except Exception:
        data = np.load(path)
        arr = data[list(data.keys())[0]] if isinstance(data, np.lib.npyio.NpzFile) else data
        arr = np.asarray(arr, dtype=np.float32)
        if vocab_size is not None and arr.ndim == 2 and arr.shape[0] != vocab_size and arr.shape[1] == vocab_size:
            arr = arr.T
        return arr.astype(np.float32)


def load_texts_if_exists(path: Path):
    if not path.exists():
        return None
    with open(path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f]


def average_token_length(texts):
    return 0.0 if texts is None or len(texts) == 0 else float(np.mean([len(t.split()) for t in texts]))


def resolve_doc_mode(dataset, train_docs):
    if DOC_EMBEDDING_MODE in ('text', 'bow_avg'):
        return DOC_EMBEDDING_MODE
    mode = DOC_MODE_BY_DATASET.get(dataset)
    if mode is not None:
        return mode
    return 'text' if average_token_length(train_docs) >= 30 else 'bow_avg'


def resolve_low_memory_batch_size(n_docs):
    min_bs = int(FASTOPIC_DEFAULTS['min_low_memory_batch_size'])
    max_bs = int(FASTOPIC_DEFAULTS['max_low_memory_batch_size'])
    if n_docs <= min_bs:
        return int(n_docs)
    return int(min(max(min_bs, int(0.2 * n_docs)), max_bs))


def normalize_rows(arr, eps=1e-12):
    arr = np.asarray(arr, dtype=np.float32)
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    return arr / np.maximum(norms, eps)


def build_doc_embeddings_from_bow(bow_csr, word_emb, normalize=True, eps=1e-12, chunk_size=20000):
    n_docs = bow_csr.shape[0]
    emb_dim = word_emb.shape[1]
    out = np.zeros((n_docs, emb_dim), dtype=np.float32)
    lengths = np.asarray(bow_csr.sum(axis=1), dtype=np.float32).reshape(-1, 1)
    lengths[lengths == 0] = 1.0
    for start in range(0, n_docs, chunk_size):
        end = min(start + chunk_size, n_docs)
        part = bow_csr[start:end].dot(word_emb)
        out[start:end] = np.asarray(part, dtype=np.float32) / lengths[start:end]
    return normalize_rows(out, eps=eps) if normalize else out


def encode_docs_with_sentence_transformer(docs, model_name, device, batch_size=128, normalize=True):
    model = SentenceTransformer(model_name, device=device)
    emb = model.encode(docs, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=normalize)
    return np.asarray(emb, dtype=np.float32)


def reduce_embeddings_with_umap(train_embeddings, test_embeddings, label):
    train_embeddings = np.asarray(train_embeddings, dtype=np.float32)
    test_embeddings = np.asarray(test_embeddings, dtype=np.float32)
    original_dim = int(train_embeddings.shape[1])
    if (not UMAP_CONFIG['enabled']) or original_dim <= int(UMAP_CONFIG['n_components']):
        return train_embeddings, test_embeddings, {'umap_enabled': False, 'umap_seconds': 0.0, 'embedding_dim_before': original_dim, 'embedding_dim_after': original_dim, 'umap_n_neighbors': 0}
    n_neighbors = min(int(UMAP_CONFIG['n_neighbors']), max(2, train_embeddings.shape[0] - 1))
    n_components = min(int(UMAP_CONFIG['n_components']), original_dim - 1 if original_dim > 2 else original_dim)
    if n_components < 2:
        return train_embeddings, test_embeddings, {'umap_enabled': False, 'umap_seconds': 0.0, 'embedding_dim_before': original_dim, 'embedding_dim_after': original_dim, 'umap_n_neighbors': n_neighbors}
    start = time.perf_counter()
    reducer = umap.UMAP(n_components=n_components, n_neighbors=n_neighbors, min_dist=float(UMAP_CONFIG['min_dist']), metric=UMAP_CONFIG['metric'], random_state=int(UMAP_CONFIG['random_state']), transform_seed=int(UMAP_CONFIG['random_state']), low_memory=True)
    train_reduced = reducer.fit_transform(train_embeddings)
    test_reduced = reducer.transform(test_embeddings)
    if UMAP_CONFIG.get('normalize_after', True):
        train_reduced = normalize_rows(train_reduced)
        test_reduced = normalize_rows(test_reduced)
    umap_seconds = time.perf_counter() - start
    return np.asarray(train_reduced, dtype=np.float32), np.asarray(test_reduced, dtype=np.float32), {'umap_enabled': True, 'umap_seconds': float(umap_seconds), 'embedding_dim_before': original_dim, 'embedding_dim_after': int(train_reduced.shape[1]), 'umap_n_neighbors': int(n_neighbors)}


class FixedPreprocess:
    def __init__(self, train_bow, vocab):
        self._train_bow = train_bow
        self._vocab = vocab

    def preprocess(self, docs):
        if len(docs) != self._train_bow.shape[0]:
            raise ValueError('docs length must match train_bow rows')
        return {'train_bow': self._train_bow, 'vocab': self._vocab}


class PassthroughDocEmbedder:
    def encode(self, docs, show_progress_bar=False, normalize_embeddings=False):
        raise RuntimeError('encode should not be called when preset_doc_embeddings is provided')


def infer_theta_from_doc_embeddings(model, doc_embeddings_np):
    doc_embeddings = torch.as_tensor(doc_embeddings_np, dtype=torch.float32)
    if not model.low_memory:
        doc_embeddings = doc_embeddings.to(model.device)
    return np.asarray(model.transform(doc_embeddings=doc_embeddings), dtype=np.float32)


def prepare_fastopic_inputs(dataset):
    if dataset in FASTOPIC_DATA_CACHE:
        payload = dict(FASTOPIC_DATA_CACHE[dataset])
        payload['cache_hit'] = True
        return payload
    data_dir = DATASET_DIRS[dataset]
    cfg = load_cfg(CONFIGS.get(dataset))
    train_bow = load_bow_csr(data_dir / 'train_bow.npz')
    test_bow = load_bow_csr(data_dir / 'test_bow.npz')
    vocab = load_vocab(data_dir / 'vocab.txt')
    vocab_size = train_bow.shape[1]
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    train_docs = load_texts_if_exists(data_dir / 'train_texts.txt')
    test_docs = load_texts_if_exists(data_dir / 'test_texts.txt')
    selected_mode = resolve_doc_mode(dataset, train_docs)
    can_use_text = train_docs is not None and test_docs is not None and len(train_docs) == train_bow.shape[0] and len(test_docs) == test_bow.shape[0]
    use_text_mode = selected_mode == 'text' and can_use_text
    start = time.perf_counter()
    source_mode = 'bow_avg'
    if use_text_mode:
        try:
            train_doc_emb = encode_docs_with_sentence_transformer(train_docs, FASTOPIC_DEFAULTS['doc_embed_model'], device=device, batch_size=int(FASTOPIC_DEFAULTS['text_batch_size']), normalize=bool(FASTOPIC_DEFAULTS['normalize_embeddings']))
            test_doc_emb = encode_docs_with_sentence_transformer(test_docs, FASTOPIC_DEFAULTS['doc_embed_model'], device=device, batch_size=int(FASTOPIC_DEFAULTS['text_batch_size']), normalize=bool(FASTOPIC_DEFAULTS['normalize_embeddings']))
            source_mode = 'text'
        except Exception:
            use_text_mode = False
    if not use_text_mode:
        word_emb = load_word_embeddings(data_dir / 'word_embeddings.npz', vocab_size=vocab_size)
        train_doc_emb = build_doc_embeddings_from_bow(train_bow, word_emb, normalize=bool(FASTOPIC_DEFAULTS['normalize_embeddings']))
        test_doc_emb = build_doc_embeddings_from_bow(test_bow, word_emb, normalize=bool(FASTOPIC_DEFAULTS['normalize_embeddings']))
    train_doc_emb, test_doc_emb, umap_meta = reduce_embeddings_with_umap(train_doc_emb, test_doc_emb, label=dataset)
    payload = {'cfg': cfg, 'train_bow': train_bow, 'vocab': vocab, 'device': device, 'source_mode': source_mode, 'train_doc_emb': train_doc_emb, 'test_doc_emb': test_doc_emb, 'umap_meta': umap_meta, 'prep_seconds': float(time.perf_counter() - start)}
    FASTOPIC_DATA_CACHE[dataset] = payload
    payload = dict(payload)
    payload['cache_hit'] = False
    return payload


## 4) Entraînement FASTopic avec embeddings reduits par UMAP


In [6]:
def train_one_fastopic(dataset, K, seed=42):
    set_seed(seed)
    total_start = time.perf_counter()
    prepared = prepare_fastopic_inputs(dataset)
    cfg = prepared['cfg']
    epochs = int(cfg.get('fastopic_epochs', FASTOPIC_DEFAULTS['epochs']))
    learning_rate = float(cfg.get('fastopic_learning_rate', FASTOPIC_DEFAULTS['learning_rate']))
    low_memory_batch_size = resolve_low_memory_batch_size(prepared['train_bow'].shape[0])
    dt_alpha = float(cfg.get('fastopic_dt_alpha', DT_ALPHA_BY_DATASET.get(dataset, FASTOPIC_DEFAULTS['dt_alpha_default'])))
    tw_alpha = float(cfg.get('fastopic_tw_alpha', FASTOPIC_DEFAULTS['tw_alpha']))
    theta_temp = float(cfg.get('fastopic_theta_temp', FASTOPIC_DEFAULTS['theta_temp']))
    preprocess = FixedPreprocess(train_bow=prepared['train_bow'], vocab=prepared['vocab'])
    model = FASTopic(num_topics=K, preprocess=preprocess, doc_embed_model=PassthroughDocEmbedder(), device=prepared['device'], low_memory=bool(FASTOPIC_DEFAULTS['low_memory']), low_memory_batch_size=low_memory_batch_size if FASTOPIC_DEFAULTS['low_memory'] else None, normalize_embeddings=False, DT_alpha=dt_alpha, TW_alpha=tw_alpha, theta_temp=theta_temp, verbose=False)
    docs_for_fit = [''] * prepared['train_bow'].shape[0]
    train_start = time.perf_counter()
    model.fit_transform(docs=docs_for_fit, epochs=epochs, learning_rate=learning_rate, preset_doc_embeddings=prepared['train_doc_emb'])
    train_seconds = time.perf_counter() - train_start
    beta = np.asarray(model.get_beta(), dtype=np.float32)
    train_theta = np.asarray(model.train_theta, dtype=np.float32)
    test_theta = infer_theta_from_doc_embeddings(model, prepared['test_doc_emb'])
    ds_out = OUTPUT_DIR / dataset
    ds_out.mkdir(parents=True, exist_ok=True)
    out_path = ds_out / f'{dataset}_{MODEL_TAG}_K{K}_seed{seed}.mat'
    scipy.io.savemat(str(out_path), {'beta': beta, 'train_theta': train_theta, 'test_theta': test_theta, 'traintheta': train_theta, 'testtheta': test_theta})
    prep_seconds = 0.0 if prepared['cache_hit'] else float(prepared['prep_seconds'])
    umap_seconds = 0.0 if prepared['cache_hit'] else float(prepared['umap_meta']['umap_seconds'])
    total_seconds = time.perf_counter() - total_start
    timing_row = {'model': MODEL_TAG, 'dataset': dataset, 'K': int(K), 'seed': int(seed), 'device': prepared['device'], 'phase': 'train', 'prep_seconds': float(prep_seconds), 'umap_seconds': float(umap_seconds), 'train_seconds': float(train_seconds), 'total_seconds': float(total_seconds), 'prep_minutes': float(prep_seconds / 60.0), 'umap_minutes': float(umap_seconds / 60.0), 'train_minutes': float(train_seconds / 60.0), 'total_minutes': float(total_seconds / 60.0), 'mode': prepared['source_mode'], 'embedding_dim_before': int(prepared['umap_meta']['embedding_dim_before']), 'embedding_dim_after': int(prepared['umap_meta']['embedding_dim_after']), 'umap_enabled': bool(prepared['umap_meta']['umap_enabled']), 'umap_n_neighbors': int(prepared['umap_meta']['umap_n_neighbors']), 'umap_cache_hit': bool(prepared['cache_hit'])}
    timing_path = ds_out / f'{dataset}_{MODEL_TAG}_K{K}_seed{seed}_timing.csv'
    pd.DataFrame([timing_row]).to_csv(timing_path, index=False)
    return {'mat_path': str(out_path), 'timing_path': str(timing_path), 'timing': timing_row}


## 5) Run


In [7]:
training_time_rows = []
for dataset in TARGET_DATASETS:
    if dataset not in DATASET_DIRS:
        continue
    for K in K_LIST:
        result = train_one_fastopic(dataset, K=K, seed=RANDOM_SEED)
        if isinstance(result, dict) and 'timing' in result:
            training_time_rows.append(result['timing'])
if training_time_rows:
    df_train_times = pd.DataFrame(training_time_rows).sort_values(['dataset', 'K']).reset_index(drop=True)
    display(df_train_times[['dataset', 'K', 'device', 'prep_seconds', 'umap_seconds', 'train_seconds', 'total_seconds', 'mode', 'embedding_dim_before', 'embedding_dim_after', 'umap_cache_hit']])
    time_csv = OUTPUT_DIR / f'{MODEL_TAG}_training_times.csv'
    df_train_times.to_csv(time_csv, index=False)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/196 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
Training FASTopic: 100%|██████████| 200/200 [04:49<00:00,  1.45s/it]
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
Training FASTopic: 100%|██████████| 200/200 [02:37<00:00,  1.27it/s]


,dataset,K,device,prep_seconds,umap_seconds,train_seconds,total_seconds,mode,embedding_dim_before,embedding_dim_after,umap_cache_hit
0,20NG,20,cuda,93.130605,71.871097,185.958524,279.450023,text,384,64,False
1,20NG,50,cuda,0.000000,0.000000,187.717869,187.766247,text,384,64,True
2,20NG,100,cuda,0.000000,0.000000,187.830556,187.910202,text,384,64,True
3,AGNews,20,cuda,48.894629,48.695751,160.679063,209.674393,bow_avg,200,64,False
4,AGNews,50,cuda,0.000000,0.000000,158.897425,158.933558,bow_avg,200,64,True
5,AGNews,100,cuda,0.000000,0.000000,158.783911,158.840431,bow_avg,200,64,True
6,IMDB,20,cuda,137.946592,94.720687,289.974731,428.683422,text,384,64,False
7,IMDB,50,cuda,0.000000,0.000000,291.858522,291.956202,text,384,64,True
8,IMDB,100,cuda,0.000000,0.000000,289.844703,290.038590,text,384,64,True
9,YahooAnswer,20,cuda,55.437509,55.257413,157.417137,212.996024,bow_avg,200,64,False


## 6) Métriques (C_V Gensim + TD + Purity + NMI)


In [8]:
import pandas as pd
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel


def topic_diversity_from_beta(beta, topn=15):
    all_words = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:topn]
        all_words.extend(top_idx.tolist())
    return len(set(all_words)) / max(1, len(all_words))


def purity_score(y_true, y_pred):
    contingency_matrix = metrics.cluster.contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)


def build_tokenized_texts_from_bow(train_bow, vocab, max_docs=10000, min_tokens=2):
    texts = []
    n_docs = min(train_bow.shape[0], max_docs)
    for i in range(n_docs):
        row = train_bow.getrow(i)
        idx = row.indices
        if len(idx) < min_tokens:
            continue
        words = [vocab[j] for j in idx if j < len(vocab)]
        if len(words) >= min_tokens:
            texts.append(words)
    return texts


def compute_cv_gensim(beta, vocab, texts, dictionary, topn=15):
    topics = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:topn]
        topic_words = [vocab[i] for i in top_idx if i < len(vocab) and vocab[i] in dictionary.token2id]
        if len(topic_words) >= 2:
            topics.append(topic_words)
    if not topics:
        return float('nan')
    cm = CoherenceModel(topics=topics, texts=texts, dictionary=dictionary, coherence='c_v')
    return float(cm.get_coherence())


rows = []
for dataset in TARGET_DATASETS:
    if dataset not in DATASET_DIRS:
        continue
    ds_out = OUTPUT_DIR / dataset
    data_dir = DATASET_DIRS[dataset]
    vocab = load_vocab(data_dir / 'vocab.txt')
    labels = np.loadtxt(data_dir / 'test_labels.txt', dtype=np.int32)
    train_bow_for_cv = load_bow_csr(data_dir / 'train_bow.npz')
    tokenized_texts = build_tokenized_texts_from_bow(train_bow_for_cv, vocab, max_docs=10000)
    dictionary = Dictionary(tokenized_texts)
    for K in K_LIST:
        mat_path = ds_out / f'{dataset}_{MODEL_TAG}_K{K}_seed{RANDOM_SEED}.mat'
        if not mat_path.exists():
            continue
        loaded = scipy.io.loadmat(str(mat_path))
        beta = loaded['beta']
        test_theta = loaded['test_theta']
        n_eval = min(len(labels), test_theta.shape[0])
        y_true = labels[:n_eval]
        y_pred = np.argmax(test_theta[:n_eval], axis=1)
        rows.append({'model': MODEL_TAG, 'dataset': dataset, 'K': K, 'C_V': compute_cv_gensim(beta, vocab, tokenized_texts, dictionary, topn=15), 'Purity': float(purity_score(y_true, y_pred)), 'NMI': float(metrics.cluster.normalized_mutual_info_score(y_true, y_pred)), 'TD_top15': float(topic_diversity_from_beta(beta, topn=15)), 'TD_top25': float(topic_diversity_from_beta(beta, topn=25))})
if rows:
    df = pd.DataFrame(rows).sort_values(['dataset', 'K']).reset_index(drop=True)
    display(df)
    csv_path = OUTPUT_DIR / f'{MODEL_TAG}_metrics_CV_TD_Purity_NMI.csv'
    df.to_csv(csv_path, index=False)


,model,dataset,K,C_V,Purity,NMI,TD_top15,TD_top25
0,Fast_topic_avec_UMAP,20NG,20,0.486282,0.379713,0.549976,0.993333,0.9900
1,Fast_topic_avec_UMAP,20NG,50,0.421150,0.403611,0.567336,0.990667,0.9696
2,Fast_topic_avec_UMAP,20NG,100,0.379997,0.478093,0.578351,0.889333,0.8264
3,Fast_topic_avec_UMAP,AGNews,20,0.481411,0.846400,0.508189,0.876667,0.8760
4,Fast_topic_avec_UMAP,AGNews,50,0.532763,0.849200,0.467893,0.568000,0.5632
5,Fast_topic_avec_UMAP,AGNews,100,0.474566,0.848800,0.466888,0.468000,0.4448
6,Fast_topic_avec_UMAP,IMDB,20,0.289939,0.700440,0.067038,0.660000,0.6440
7,Fast_topic_avec_UMAP,IMDB,50,0.295311,0.701240,0.073522,0.578667,0.5552
8,Fast_topic_avec_UMAP,IMDB,100,0.329143,0.699120,0.070180,0.512000,0.4560
9,Fast_topic_avec_UMAP,YahooAnswer,20,0.554363,0.541200,0.328690,0.933333,0.9300
